> ### Note on Labs and Assigments:
>
> 🔧 Look for the **wrench emoji** 🔧 — it highlights where you're expected to take action!
>
> These sections are graded and are not optional.
>

# **IS 4487 LAB 7: DATA TRANSFORMATION**

## Outline

- Load and preview the cleaned Megatelco dataset  
- Engineer new columns from existing data  
- Binning to simplify numeric variable values  
- On-hot-Encoding and integer encoding to change categorical into numeric values
- Use log scaling, normalization and standardization
- Try your own transformation logic  

This lab continues from **Lab 6**, where we cleaned the Megatelco dataset.  

Now, we will create new, more useful features for modeling and exploration.

<a href="https://colab.research.google.com/github/vandanara/UofUtah_IS4487/blob/main/Labs/lab_07_data_transformation.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

If you're new to Colab: [Colab FAQ](https://research.google.com/colaboratory/faq.html)




## Megatelco Data Dictionary

 DEMOGRAPHIC VARIABLES:
 - College - has the customer attended some college (one, zero)
 - Income - annual income of customer
 - House - estimated price of the customer's home (if applicable)

 USAGE VARIABLES:
 - Data Overage Mb - Average number of megabytes that the customer used in excess of the plan limit (over last 12 months)
 - Data Leftover Mb - Average number of megabytes that the customer use was below the plan limit (over last 12 months)
 - Data Mb Used - Average number of megabytes used per month (over last 12 months)
 - Text Message Count - Average number of texts per month (over last 12 months)
 - Over 15 Minute Calls Per Month - Average number of calls over 15 minutes in duration per month (over last 12 months)
 - Average Call Duration- Average call duration (over last 12 months)

PHONE VARIABLES:
 - Operating System - Current operating system of phone
 - Handset Price - Retail price of the phone used by the customer

ATTITUDINAL VARIABLES:
 - Reported Satisfaction - Survey response to "How satisfied are you with your current phone plan?" (high, med, low)
 - Reported Usage Level - Survey response to "How much do your use your phone?" (high, med, low)
 - Considering Change of Plan - Survey response to "Are you currently planning to change companies when your contract expires?" (high, med, low)

OTHER VARIABLES
 - Leave - Did this customer churn with the last contract expiration? (LEAVE, STAY)
 - ID - Customer identifier

## **Part 1: Load Cleaned Data from Lab 6 and Preview it**

In this part of the lab, we will load the cleaned data from Lab 6. We had previously completed the following cleaning steps.

- Cleaned column names
- Fixed data types
- Handled missing values
- Removed duplicate records
- Reviewed for outliers

The link to the cleaned data is provided for your below. It contains 14,200 rows instead of 15,016 that we started out with in Lab 6.

In [256]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/vandanara/UofUtah_IS4487/refs/heads/main/DataSets/megatelco_cleaned.csv"
df = pd.read_csv(url)

df.sample(n =5)

,id,college,income,data_overage_mb,data_leftover_mb,data_mb_used,text_message_count,house,handset_price,over_15mins_calls_per_month,average_call_duration,reported_satisfaction,reported_usage_level,considering_change_of_plan,leave,operating_system
14122,21704,one,299942.00,346,54.0,3304.0,102,1140269,898.0,7.0,9.0,low,low,yes,STAY,IOS
4418,8404,one,55805.89,49,24.0,5102.0,154,1043182,1215.0,1.0,2.0,avg,low,yes,LEAVE,IOS
9941,12479,one,316785.00,54,66.0,3718.0,144,860692,1217.0,3.0,15.0,low,low,no,LEAVE,IOS
12746,15829,zero,115182.00,137,7.0,2995.0,67,830839,899.0,0.0,7.0,low,low,unknown,STAY,IOS
10847,17182,one,264757.00,233,20.0,2321.0,111,518265,803.0,16.0,10.0,low,low,yes,LEAVE,IOS


In [252]:
#check datatypes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14200 entries, 0 to 14199
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id                           14200 non-null  int64  
 1   college                      14200 non-null  object 
 2   income                       14200 non-null  float64
 3   data_overage_mb              14200 non-null  int64  
 4   data_leftover_mb             14200 non-null  float64
 5   data_mb_used                 14200 non-null  float64
 6   text_message_count           14200 non-null  int64  
 7   house                        14200 non-null  int64  
 8   handset_price                14200 non-null  float64
 9   over_15mins_calls_per_month  14200 non-null  float64
 10  average_call_duration        14200 non-null  float64
 11  reported_satisfaction        14200 non-null  object 
 12  reported_usage_level         14200 non-null  object 
 13  considering_chan

The only cleaning step we have to redo again is fixing data types. When we read in data, it once again infers data types.

In [253]:
# Check original data types
print("Original dtypes:\n", df.dtypes)

# Convert object to nomimal categorical - can use df[colname].astype() to convert to nominal categorical
obj_to_nomcat_cols = ['considering_change_of_plan', 'college', 'operating_system', 'leave']
for acol in obj_to_nomcat_cols:
    df[acol] = df[acol].astype('category')

# Convert object/text columns with limited possible values with an order to ordinal categorical columnns
obj_to_ordcat_cols = ['reported_satisfaction', 'reported_usage_level']
for acol in obj_to_ordcat_cols:
    df[acol] = pd.Categorical(df[acol], categories = ['low', 'avg', 'high'], ordered = True)

# Check updated data types
print("\nUpdated dtypes:\n", df.dtypes)



Original dtypes:
 id                               int64
college                         object
income                         float64
data_overage_mb                  int64
data_leftover_mb               float64
data_mb_used                   float64
text_message_count               int64
house                            int64
handset_price                  float64
over_15mins_calls_per_month    float64
average_call_duration          float64
reported_satisfaction           object
reported_usage_level            object
considering_change_of_plan      object
leave                           object
operating_system                object
dtype: object

Updated dtypes:
 id                                int64
college                        category
income                          float64
data_overage_mb                   int64
data_leftover_mb                float64
data_mb_used                    float64
text_message_count                int64
house                             int64
handse

In [255]:
# View missing value counts
print("Missing values per column:\n", df.isnull().sum())

# Check for exact duplicates
print(f"\nNumber of duplicate rows: {df.duplicated().sum()}")


Missing values per column:
 id                             0
college                        0
income                         0
data_overage_mb                0
data_leftover_mb               0
data_mb_used                   0
text_message_count             0
house                          0
handset_price                  0
over_15mins_calls_per_month    0
average_call_duration          0
reported_satisfaction          0
reported_usage_level           0
considering_change_of_plan     0
leave                          0
operating_system               0
dtype: int64

Number of duplicate rows: 0


## **Part 2: Creating New Features**

**Data cleaning** makes the data more ***usable***.

After cleaning, the major next step is data transformation, getting data ready for predictive modeling and **feature engineering** — creating new columns from data to better capture useful patterns, and make it more ***useful*** and ready for Machine Learning.

In this and the next few sections, we will try three common methods:

1. Creating new columns using builtin functions and algebra (+ - / *)
2. Encoding using either one-hot-encoding `pd.get_dummies()` for nominal catgeorical variables (e.g., satisfaction levels) or integer encoding `df.map()` for ordinal categorical variables
3. Binning continuous/ numeric variables using `pd.cut()` or `pd.qcut()` — to group them into bins (quantiles)

These new features help ML  models learn better, and thus makes these models more powerful (more predictive power), and may make results easier to interpret.

Things to think about:
- What are some new combinations (e.g., ratios, additions) of the existing columns that might be useful and meaningful to predict whether a customer would leave or stay?
- Can you create flag variables (a binary 0/1 variable) to highlight important traits?


In [ ]:
# Create a total data usage variable (used + leftover)
df['total_data_mb'] = df['data_mb_used'] + df['data_leftover_mb']

# Create a ratio of overage to used data
df['overage_ratio'] = df['data_overage_mb'] / (df['data_mb_used'] + 1)  # add 1 to avoid divide-by-zero

# Create a binary flag for high texters (over 135 texts which is the mean)
df['high_texter'] = (df['text_message_count'] > 135).astype(int)

# Preview new columns
df[['total_data_mb', 'overage_ratio', 'high_texter']].head()


### 🔧 **Try It Yourself  Part 2**

2.1. Create a variable called `call_volume` by multiplying `over_15mins_calls_per_month` by `average_call_duration`

2.2. Create a binary flag `high_data_user` for users where `data_mb_used` is above the median

2.3. Use `.head()` to check your new columns



In [ ]:
# 🔧 Add code here

## Part 3: Binning Continous Variables

Binning is the process of grouping numeric variables into categories (e.g., "low", "medium", "high").

### Why We Bin:
- Helps reduce the impact of outliers
- Allows us to use numeric values in models that prefer categories
- Simplifies interpretation and visualization

### Things to think about:
- Would grouping values make patterns more visible?
- Do we want equal-sized groups or logical cutoffs?
- Is the variable skewed?

**Tools:**  
- `pd.qcut()` for quantile-based bins (equal frequency)  
- `pd.cut()` for equal-width or custom bins


In [ ]:
# Bin income into 3 groups (quantiles): Low, Medium, High
df['income_group'] = pd.qcut(df['income'], q=3, labels=['Low', 'Medium', 'High'])

# Bin average call duration into quartiles (labels as integers)
df['call_duration_group'] = pd.qcut(df['average_call_duration'], q=4, labels=False)

# Preview new groupings
df[['income', 'income_group', 'average_call_duration', 'call_duration_group']].head()

### 🔧 Try It Yourself – Part 3

1. Use `pd.cut()` to group `data_mb_used` into 3 labeled bins: "Light", "Moderate", "Heavy"
2. Use `pd.qcut()` on `text_message_count` to split into 4 equal-sized groups
3. Print `.value_counts()` on each new column to see how values are distributed

In [ ]:
# 🔧 Add code here

## Part 4: Scaling Numeric Variables

Scaling transforms values to a common range (often 0–1), which helps many machine learning models perform better.

### When to Scale:
- When features have very different ranges (e.g., income vs. call duration)
- When using distance-based models (e.g., KNN, SVM)
- When comparing magnitudes across features

### Common Methods:
- `MinMaxScaler`: scales to 0–1 range
- `StandardScaler`: centers data around 0 with unit variance

### Things to think about:
- Are features on different scales?
- Does my algorithm care about magnitude?

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Choose columns to scale
cols_to_scale = ['income', 'data_mb_used', 'average_call_duration']

# Initialize and apply scaler
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df[cols_to_scale])

# Add scaled columns back to df
df['income_scaled'] = df_scaled[:, 0]
df['data_mb_used_scaled'] = df_scaled[:, 1]
df['avg_call_dur_scaled'] = df_scaled[:, 2]

# Preview
df[['income_scaled', 'data_mb_used_scaled', 'avg_call_dur_scaled']].head()

### 🔧 Try It Yourself – – Part 4

1. Scale the `handset_price` and `over_15mins_calls_per_month` columns using `MinMaxScaler`
2. Add the scaled values back to the dataframe using 2 new columns with suffix `_scaled`
3. Use `.describe()` to compare original vs. scaled versions and make a comment on what you observe


In [ ]:
# 🔧 Add code here

Add comment here

## Part 5: Encoding Categorical Variables

Most machine learning models can't handle string categories directly—so we convert them into numbers using **encoding**.

### Types of Encoding:
- **One-hot encoding**: creates a binary column for each category (preferred for nominal variables)
- **Ordinal encoding**: assigns integers (use only for ordered categories)

### Things to consider:
- Is the variable nominal (e.g., OS type) or ordinal (e.g., satisfaction)?
- How many unique categories are there?
- Will one-hot encoding make the dataset too wide?

**Tool:** `pd.get_dummies()`

In [ ]:
# One-hot encode 'reported_usage_level'
df_encoded = pd.get_dummies(df, columns=['reported_usage_level'], prefix='usage')

# One-hot encode 'income_group'
df_encoded = pd.get_dummies(df_encoded, columns=['income_group'], prefix='income')

# Preview new columns
df_encoded.filter(like='usage_').head()

### 🔧 Try It Yourself – Part 5

1. One-hot encode `reported_satisfaction` and `operating_system`
2. Print `.shape` of your dataframe before and after to observe any big changes
3. How many new columns were added?


In [ ]:
# 🔧 Add code here

🔧 Add comment here:

# 🔧 Part 6: Reflection (100 words or less per question)

1. Which transformation do you think had the biggest impact on preparing your data for modeling?
2. Are there any features you created that you think will be especially useful for predicting churn?

🔧 Add comment here:

## Export Your Notebook to Submit in Canvas
- Use the instructions from Lab 1

In [ ]:
!jupyter nbconvert --to html "lab_07_LastnameFirstname.ipynb"